# 🌐 Universal Tutorial Scraper
Works on **any website** — no hardcoded HTML selectors.  
Just upload your seed `.xlsx` and set `BASE_URL`. Everything else is automatic.

### Bugs fixed vs original
| # | Bug | Fix |
|---|---|---|
| 1 | `KeyError: 'Tutorials'` — column lowercased then accessed with capital T | Auto-detect slug column after normalising |
| 2 | `df["Tutorials"][i]` in scrape loop — same KeyError again | Use `slug_col` variable everywhere |
| 3 | `requests` imported nowhere | Added `import requests` |
| 4 | Seeds file uses `url` column — was being ignored | If `url` column exists, use those directly instead of building URLs |
| 5 | `URL_list.index(url)` — crashes if duplicates exist | Use `enumerate` with index instead |
| 6 | Output filename breaks if seeds file has no `_seeds` in name | Safer output name logic |

---
## Cell 1 — Install Dependencies

In [ ]:
!pip install -q readability-lxml lxml_html_clean openpyxl
print("✅ Dependencies ready")

---
## Cell 2 — Imports & Configuration

In [ ]:
import requests
import time
import re
import os
from urllib.robotparser import RobotFileParser
from urllib.parse import urlparse

import pandas as pd
from bs4 import BeautifulSoup
from readability import Document

# ── CONFIGURATION — change these per site ───────────────────────────────────
XLSX_FILE  = "groklearning_seeds.xlsx"   # your uploaded seeds file
BASE_URL   = "https://groklearning.com/" # base URL of the site
URL_SUFFIX = ""                          # suffix after slug, e.g. "/" or ".asp"
# ────────────────────────────────────────────────────────────────────────────

print("✅ Imports ready")

---
## Cell 3 — Load Seeds File
**Fix 1 & 4:** Auto-detects the slug column regardless of capitalisation.  
If your file already has a `url` column (like `groklearning_seeds.xlsx`), those URLs are used directly — no URL building needed.

In [ ]:
from google.colab import files

uploaded = files.upload()            # file picker
XLSX_FILE = list(uploaded.keys())[0]

df = pd.read_excel(XLSX_FILE, engine="openpyxl")

# Normalise column names ONCE — lowercase + strip
df.columns = [c.strip().lower() for c in df.columns]

print(f"Loaded {len(df)} rows from '{XLSX_FILE}'")
print("Columns:", df.columns.tolist())
df.head()

---
## Cell 4 — Build URL List
**Fix 4:** If a `url` column already exists, use it directly.

In [ ]:
# Auto-detect slug column — works with 'slug', 'tutorials', 'tutorial', 'course', etc.
preferred_slug_cols = ["slug", "tutorials", "tutorial", "course", "name"]
slug_col = next((c for c in preferred_slug_cols if c in df.columns), df.columns[0])
print(f"Using slug column: '{slug_col}'")

# If a ready-made 'url' column exists, use it directly — no URL building needed
if "url" in df.columns:
    URL_list = df["url"].dropna().astype(str).str.strip().tolist()
    slug_list_raw = df[slug_col].astype(str).str.strip().tolist()
    print("✅ Using pre-built URLs from 'url' column")
else:
    slug_list_raw = df[slug_col].dropna().astype(str).str.strip().tolist()
    URL_list = [BASE_URL + slug + URL_SUFFIX for slug in slug_list_raw]
    print("✅ URLs built from BASE_URL + slug")

print(f"\nFirst 5 URLs:")
for u in URL_list[:5]:
    print(" ", u)

---
## Cell 5 — Robots.txt Check

In [ ]:
parsed     = urlparse(BASE_URL)
robots_url = f"{parsed.scheme}://{parsed.netloc}/robots.txt"

rp = RobotFileParser()
rp.set_url(robots_url)
try:
    rp.read()
    print(f"✅ robots.txt read: {robots_url}")
except Exception as e:
    print(f"⚠️  Could not read robots.txt ({e}) — allowing all URLs")
    rp = None

MY_AGENT = "Mozilla/5.0"

allowed_URLs = []
blocked_URLs = []
for url in URL_list:
    if rp is None or rp.can_fetch(MY_AGENT, url):
        allowed_URLs.append(url)
    else:
        blocked_URLs.append(url)

# If everything blocked, likely a robots.txt parsing edge case — override
if len(allowed_URLs) == 0 and len(blocked_URLs) > 0:
    print("⚠️  All URLs blocked by can_fetch() — likely a parsing edge case. Overriding.")
    allowed_URLs = URL_list
    blocked_URLs = []

crawl_delay = (rp.crawl_delay(MY_AGENT) if rp else None) or 2
print(f"\n✅ Allowed  : {len(allowed_URLs)}")
print(f"🚫 Blocked  : {len(blocked_URLs)}")
print(f"⏱  Delay    : {crawl_delay}s")

---
## Cell 6 — Universal Extraction Helpers

In [ ]:
def extract_main_content(soup, html):
    """Uses readability to find the main article body on ANY site."""
    try:
        doc = Document(html)
        summary_soup = BeautifulSoup(doc.summary(), "html.parser")
        return summary_soup.get_text(separator=" ", strip=True)
    except:
        return soup.get_text(separator=" ", strip=True)[:2000]

def extract_title(soup):
    for tag in ["h1", "title"]:
        found = soup.find(tag)
        if found:
            return found.get_text(strip=True)
    return "nan"

def extract_intro(soup):
    """First meaningful paragraph (>40 chars)."""
    for p in soup.find_all("p"):
        text = p.get_text(strip=True)
        if len(text) > 40:
            return text[:300]
    return "nan"

def count_code_blocks(soup):
    return len(soup.find_all(["code", "pre"]))

def extract_headings(soup):
    h2s = soup.find_all("h2")
    return ", ".join([h.get_text(strip=True) for h in h2s[:8]])

def count_words(text):
    return len(re.findall(r"\w+", text))

def extract_links(soup, base):
    domain   = urlparse(base).netloc
    links    = soup.find_all("a", href=True)
    internal = [l for l in links if domain in l["href"] or l["href"].startswith("/")]
    return len(internal)

print("✅ Helpers ready")

---
## Cell 7 — Scrape Loop
**Fix 2:** Uses `slug_col` and `enumerate` — no more `KeyError` or `.index()` crashes.

In [ ]:
from IPython.display import clear_output

records = []

headers = {
    "User-Agent"               : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36",
    "Accept"                   : "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language"          : "en-US,en;q=0.9",
    "Accept-Encoding"          : "gzip, deflate, br",
    "Connection"               : "keep-alive",
    "Upgrade-Insecure-Requests": "1",
    "Referer"                  : "https://www.google.com/",
}

total = len(allowed_URLs)

# FIX 5: use enumerate — avoids URL_list.index() which crashes on duplicates
for j, url in enumerate(allowed_URLs):
    # Recover slug from the original df row that matches this URL
    if "url" in df.columns:
        match = df[df["url"] == url]
        slug  = match[slug_col].values[0] if len(match) > 0 else url
    else:
        slug = slug_list_raw[j] if j < len(slug_list_raw) else url

    # Progress bar
    pct = (j + 1) / total
    bar = "█" * int(pct * 30) + "░" * (30 - int(pct * 30))
    clear_output(wait=True)
    print(f"[{bar}] {j+1}/{total}")
    print(f"  Current : {slug}")
    print(f"  OK: {sum(1 for r in records if r['status']=='ok')}  "
          f"Failed: {sum(1 for r in records if r['status']!='ok')}")

    time.sleep(crawl_delay)

    try:
        r = requests.get(url, headers=headers, timeout=15)
        r.raise_for_status()
        html = r.text
        soup = BeautifulSoup(html, "html.parser")

        content = extract_main_content(soup, html)
        records.append({
            "Slug"           : slug,
            "URL"            : url,
            "Title"          : extract_title(soup),
            "Intro"          : extract_intro(soup),
            "Content Preview": content[:1000],
            "Word Count"     : count_words(content),
            "Code Blocks"    : count_code_blocks(soup),
            "Headings"       : extract_headings(soup),
            "Internal Links" : extract_links(soup, url),
            "status"         : "ok",
        })

    except Exception as e:
        records.append({
            "Slug": slug, "URL": url, "Title": "nan", "Intro": "nan",
            "Content Preview": "nan", "Word Count": 0, "Code Blocks": 0,
            "Headings": "nan", "Internal Links": 0,
            "status": f"failed: {e}",
        })

clear_output(wait=True)
print("━" * 45)
print(f"✅ Done! {sum(1 for r in records if r['status']=='ok')} OK  "
      f"| {sum(1 for r in records if r['status']!='ok')} failed")
print("━" * 45)

---
## Cell 8 — Build DataFrame & Preview

In [ ]:
Final_df = pd.DataFrame(records)

print(f"Total rows  : {len(Final_df)}")
print(f"Successful  : {(Final_df['status']=='ok').sum()}")
print(f"Failed      : {(Final_df['status']!='ok').sum()}")
Final_df

---
## Cell 9 — Save & Download
**Fix 6:** Output name works even if the seeds file has no `_seeds` in its name.

In [ ]:
from google.colab import files

# FIX 6: safe output name — strip _seeds if present, always append _output
base_name   = os.path.splitext(XLSX_FILE)[0]          # drop .xlsx
base_name   = re.sub(r"_seeds$", "", base_name)       # drop _seeds suffix if present
output_name = base_name + "_output.xlsx"

Final_df.to_excel(output_name, index=False)
print(f"✅ Saved → {output_name}")
files.download(output_name)